# Eksplorasi Dataset

Notebook ini mendokumentasikan perjalanan dataset dari CSV mentah sampai dataset
yang siap dipakai untuk pemodelan. Fokusnya adalah kualitas data: nilai kosong,
ID duplikat, kelengkapan teks, kelengkapan metadata, dan karakteristik panjang teks.

Setiap langkah yang mengubah jumlah baris dicatat di sini, supaya angka pada
penelitian dapat ditelusuri asalnya.

Yang **tidak** termasuk di notebook ini:

- filter bahasa, dibahas di `02_language_filtration.ipynb`
- pembentukan fitur semantik, encoding SBERT, dan analisis top-K, dibahas di
  `03_semantic_feature.ipynb`

Ruang lingkup notebook ini:

1. audit dataset mentah
2. deduplikasi ID
3. pembentukan `semantic_text`
4. audit panjang token
5. penyusunan dataset eksperimen dan dataset pemodelan


## Alur Dataset

Tabel berikut merangkum setiap tahap dan siapa yang mengerjakannya.

| Tahap | Jumlah baris | File | Dibuat di |
|---|---:|---|---|
| Dataset mentah | 443.610 | `data/raw/TMDB  IMDB Movies Dataset.csv` | notebook 01 |
| Master setelah deduplikasi | 429.755 | `data/processed/movies.parquet` | notebook 01 |
| Dataset eksperimen | 396.254 | `data/processed/movies_experiment.parquet` | notebook 01 |
| Dataset final English-only | 388.241 | `data/processed/movies_final.parquet` | notebook 02 |
| Dataset pemodelan | 387.188 | `data/processed/movies_modeling.parquet` | notebook 01, bagian 13 |

Empat pengurangan pada alur ini punya sebab berbeda, dan itu sebabnya keempatnya
dipisah ke tahap sendiri.

Dari 443.610 ke 429.755, yang dihapus adalah baris dengan ID kembar. Barisnya tidak
identik, jadi pemilihan dilakukan dengan aturan kelengkapan metadata.

Dari 429.755 ke 396.254, yang dihapus adalah film yang ketiga kolom teksnya kosong.
Film seperti ini tidak punya bahan untuk di-encode, sehingga tidak bisa masuk
pipeline semantik.

Dari 396.254 ke 388.241, yang dihapus adalah film dengan overview non-Inggris dan
overview kosong. Tahap ini dibahas di notebook 02.

Terakhir, dari 388.241 ke 387.188, yang dihapus adalah film yang keempat kolom
metadatanya kosong semua. Tahap ini ada di bagian 13 notebook ini.


## 1. Load Dataset Mentah dan Audit Ringkasan


Cell berikut memuat CSV mentah, menjalankan validasi, lalu mencetak ringkasan awal.

`load_raw_movies()` membaca file dan menolak path yang salah atau file kosong.
`validate_movies()` hanya mengamati data, tidak mengubahnya. Laporan yang
dihasilkan memuat jumlah baris, jumlah kolom, jumlah ID unik, jumlah grup ID
duplikat, dan jumlah ID kosong.


In [17]:
from movie_recommender.data.loader import load_raw_movies
from movie_recommender.data.validator import validate_movies
from movie_recommender.utils.paths import DATA_RAW_DIR

RAW_PATH = DATA_RAW_DIR / "TMDB  IMDB Movies Dataset.csv"

movies = load_raw_movies(RAW_PATH)
report = validate_movies(movies)

print("rows:", report["n_rows"])
print("columns:", report["n_columns"])
print("unique ids:", report["n_unique_ids"])
print("duplicate id groups:", report["n_duplicate_ids"])
print("null ids:", report["n_null_ids"])

rows: 443610
columns: 29
unique ids: 429755
duplicate id groups: 3463
null ids: 0


Hasil di atas menunjukkan tiga hal yang perlu dicatat.

Pertama, dataset mentah berisi 443.610 baris tetapi hanya 429.755 ID unik. Artinya
ada 13.855 baris berlebih yang tersebar di 3.463 grup ID duplikat.

Kedua, tidak ada ID kosong. Ini penting karena ID akan dipakai sebagai kunci utama
di seluruh pipeline berikutnya.

Ketiga, file CSV memuat 29 kolom, sementara kebutuhan penelitian hanya sekitar
sebelas kolom. Kolom sisanya tetap dibaca untuk keperluan audit, tetapi tidak
dipakai sebagai fitur.

Variabel `movies` dan `report` dari cell ini dipakai oleh seluruh bagian setelahnya.


## 2. Tabel Missing dan Empty Seluruh Kolom


Ini menampilkan tabel kolom yang punya nilai kosong. Tabelnya membedakan:
- n_missing: nilai NaN.
- n_empty: string kosong atau whitespace.
- n_blank: total keduanya.
- pct_blank: persentase kosong dari jumlah baris.

In [18]:
report["missing"]

,n_missing,n_empty,n_blank,pct_blank
release_date,25344,0,25344,5.71
backdrop_path,256596,0,256596,57.84
homepage,387845,3,387848,87.43
overview,44973,64,45037,10.15
poster_path,79352,0,79352,17.89
tagline,349602,0,349602,78.81
genres,83793,0,83793,18.89
production_companies,180228,0,180228,40.63
production_countries,121839,0,121839,27.47
spoken_languages,109749,0,109749,24.74


Tabel di atas hanya menampilkan kolom yang punya nilai kosong. Dua kolom yang paling
relevan untuk penelitian adalah `overview` dan `tagline`.

`overview` kosong pada 45.037 baris, atau 10,15 persen. Angka ini terdiri dari
44.973 nilai NaN dan 64 string yang hanya berisi spasi. Perbedaan ini penting:
kalau hanya memeriksa NaN, 64 baris tadi akan terlewat.

`tagline` kosong pada 349.602 baris, atau 78,81 persen. Artinya tagline tidak bisa
dijadikan sumber utama teks, hanya pelengkap.

`homepage` punya persentase kosong tertinggi, 87,43 persen. Kolom ini tidak dipakai
di penelitian, jadi tidak menjadi masalah.


## 3. Audit Kolom yang Relevan untuk Penelitian


Cell ini membuat tabel dari report["relevant_stats"], lalu .T membalik tabelnya agar nama kolom, seperti overview, tagline, dan keywords, jadi baris.

Tujuannya fokus melihat kondisi tujuh kolom yang dipakai dalam rancangan fitur teks dan metadata, bukan seluruh 29 kolom.

In [19]:
import pandas as pd

pd.DataFrame(report["relevant_stats"]).T

,n_missing,n_empty
overview,44973,64
tagline,349602,0
keywords,271970,0
genres,83793,0
directors,10003,0
writers,64952,0
cast,69641,0


## 4. Contoh Overview yang Hanya Berisi Whitespace


Contoh di atas memperlihatkan baris yang secara teknis bukan NaN, tetapi isinya
tidak berguna. Kalau hanya memakai `isna()`, baris seperti ini akan dianggap
terisi dan lolos ke tahap berikutnya.

Karena itu, seluruh pemeriksaan kosong pada notebook ini memakai definisi yang
sama: NaN, string kosong, atau string yang hanya berisi spasi.


In [20]:
blank_overview = movies[
    movies["overview"].notna()
    & movies["overview"].astype(str).str.strip().eq("")
]

blank_overview[["id", "title", "overview", "tagline", "keywords"]].head()

,id,title,overview,tagline,keywords
146479,64798,Garrison,,NaN,NaN
240639,1169535,We Were Fine,,NaN,NaN
249240,1157723,Anna,,NaN,NaN
269927,1373367,Notes on Blindness: Rainfall,,NaN,NaN
270959,1378522,C.K.O.D.2 - Evacuation Plan,,NaN,NaN


## 5. Contoh Baris dengan ID Duplikat


Tabel di atas memperlihatkan baris dengan ID yang sama. Perhatikan kolom `cast`:
dua baris dengan ID identik bisa punya daftar aktor yang berbeda, dan salah satunya
lebih lengkap.

Inilah alasan deduplikasi tidak boleh memakai `keep="first"`. Memilih baris pertama
secara sembarang berisiko membuang metadata yang lebih lengkap.

Bagian berikutnya menetapkan aturan pemilihan yang eksplisit.


In [21]:
duplicate_ids = movies[movies["id"].duplicated(keep=False)]

duplicate_ids.sort_values("id").head(10)[
    ["id", "title", "overview", "genres", "cast"]
]

,id,title,overview,genres,cast
14238,2979,Return of the Ghostbusters,The mile high city is rocked when an ancient E...,"Comedy, Fantasy, Science Fiction",NaN
14239,2979,Return of the Ghostbusters,The mile high city is rocked when an ancient E...,"Comedy, Fantasy, Science Fiction","Tim Johnson, J. Michael Weiss, Joel Hanson, Ro..."
22370,5804,The Key to Reserva,Finding an unfinished script written by Alfred...,"Thriller, Comedy","Simon Baker, Kelli O'Hara, Michael Stuhlbarg, ..."
22369,5804,The Key to Reserva,Finding an unfinished script written by Alfred...,"Thriller, Comedy","Simon Baker, Kelli O'Hara, Michael Stuhlbarg, ..."
22372,5804,The Key to Reserva,Finding an unfinished script written by Alfred...,"Thriller, Comedy","Simon Baker, Kelli O'Hara, Michael Stuhlbarg, ..."
22371,5804,The Key to Reserva,Finding an unfinished script written by Alfred...,"Thriller, Comedy",NaN
172203,6080,Death on the Nile: Making of Featurette,Featurette documenting the making of the film ...,Documentary,NaN
172202,6080,Death on the Nile: Making of Featurette,Featurette documenting the making of the film ...,Documentary,"Jane Birkin, Lois Chiles, Bette Davis, Mia Far..."
20666,6962,The Hunt for the Hidden Relic,"STEFFEN VOGT, aged around 25, is helping out a...","Adventure, Science Fiction, Fantasy, Thriller","Matthias Koeberlin, Naike Rivelli, Manou Lubow..."
20667,6962,The Hunt for the Hidden Relic,"STEFFEN VOGT, aged around 25, is helping out a...","Adventure, Science Fiction, Fantasy, Thriller","Matthias Koeberlin, Manou Lubowski, Naike Rive..."


## 6. Deduplikasi dan Pemeriksaan Hasil


Aturan di atas menghasilkan 429.755 baris, turun 13.855 dari dataset mentah. Angka
ini sama dengan jumlah ID unik pada audit pertama, jadi seluruh duplikat berhasil
dihapus tanpa ada film yang hilang.

Perlu dicatat bahwa aturan ini adalah **heuristik pemilihan record**, bukan
penilaian kualitas film. Tidak ada rating atau popularitas yang dilibatkan, karena
tujuan penelitian bukan memilih film terbaik, melainkan memilih baris dengan
informasi terlengkap.


In [22]:
from movie_recommender.data.deduplicate import deduplicate_movies

movies_dedup = deduplicate_movies(movies)

print("before:", movies.shape)
print("after:", movies_dedup.shape)
print("duplicate IDs:", movies_dedup["id"].duplicated().sum())

before: (443610, 29)
after: (429755, 29)
duplicate IDs: 0


## 7. Menyimpan Master Dataset ke Parquet


Parquet dipilih karena dua alasan. Ukurannya lebih kecil daripada CSV untuk data
yang sama, dan tipe data tetap terjaga tanpa perlu parsing ulang saat dibaca.

Folder tujuan dibuat lebih dulu agar cell ini tidak gagal pada mesin yang baru
melakukan clone repository.


In [23]:
from movie_recommender.utils.paths import MOVIES_PARQUET

MOVIES_PARQUET.parent.mkdir(parents=True, exist_ok=True)

movies_dedup.to_parquet(
    MOVIES_PARQUET,
    index=False,
    compression="zstd",
)

### 7.1 Verifikasi Parquet Setelah Dibaca Ulang

File Parquet dibaca kembali lalu dibandingkan dengan DataFrame sebelum disimpan. `assert_frame_equal` memeriksa isi, nama kolom, urutan, dan tipe data. Kalau tidak ada error, proses simpan-baca dianggap konsisten.


In [24]:
reloaded = pd.read_parquet(MOVIES_PARQUET)

pd.testing.assert_frame_equal(reloaded, movies_dedup)

## 8. Ketersediaan Teks dan Metadata


Bagian ini menjawab satu pertanyaan penting: berapa banyak film yang benar-benar
bisa diproses oleh pipeline semantik.

Hasilnya, 396.254 film punya minimal satu kolom teks terisi, dan 33.501 film tidak
punya teks sama sekali. Dari 33.501 itu, 33.405 masih punya metadata, sehingga
secara teori masih bisa direpresentasikan lewat jalur metadata. Sisanya, 96 film,
tidak punya teks maupun metadata.

Angka 396.254 inilah yang menjadi populasi dataset eksperimen, dan menjadi batas
atas jumlah embedding yang bisa dibuat.


In [25]:
text_columns = ["overview", "tagline", "keywords"]
metadata_columns = ["genres", "directors", "writers", "cast"]

has_semantic_text = (
    movies_dedup[text_columns]
    .fillna("")
    .astype(str)
    .apply(lambda series: series.str.strip().ne(""))
    .any(axis=1)
)

has_any_metadata = (
    movies_dedup[metadata_columns]
    .fillna("")
    .astype(str)
    .apply(lambda series: series.str.strip().ne(""))
    .any(axis=1)
)

print("unique movies:", len(movies_dedup))
print("semantic eligible:", int(has_semantic_text.sum()))
print("no semantic text:", int((~has_semantic_text).sum()))
print(
    "no semantic text but has metadata:",
    int(((~has_semantic_text) & has_any_metadata).sum()),
)
print(
    "no semantic text and no metadata:",
    int(((~has_semantic_text) & (~has_any_metadata)).sum()),
)

unique movies: 429755
semantic eligible: 396254
no semantic text: 33501
no semantic text but has metadata: 33405
no semantic text and no metadata: 96


## 9. Membentuk semantic_text

Bagian ini mengubah tiga kolom teks menjadi satu kolom turunan yang siap
di-encode, lalu memeriksa hasilnya secara manual pada beberapa contoh.


`build_semantic_text()` menggabungkan `overview`, `tagline`, dan `keywords` menjadi
satu teks berlabel. Formatnya:

```text
Overview: {overview}

Tagline: {tagline}

Keywords: {keywords}
```

Label dipertahankan supaya encoder mengetahui asal setiap bagian. Field yang kosong
dilewati, bukan menghasilkan label tanpa isi.

Punctuation, stopwords, dan negasi tidak dihapus. Teks tidak di-lowercase, dan tidak
ada stemming atau lemmatization. Nilai tagline yang persis `none`, `n/a`, atau `-`
diperlakukan sebagai placeholder kosong.


In [27]:
from movie_recommender.preprocessing.text import build_semantic_text

semantic_text = build_semantic_text(movies_dedup)

print("same number of rows:", len(semantic_text) == len(movies_dedup))
print("same index:", semantic_text.index.equals(movies_dedup.index))
print("empty semantic texts:", int(semantic_text.eq("").sum()))

same number of rows: True
same index: True
empty semantic texts: 33501


Contoh di atas dipakai untuk memeriksa apakah label `Overview:`, `Tagline:`, dan
`Keywords:` muncul sesuai field yang tersedia. Teks lengkap tidak ditampilkan agar
tabel tetap terbaca.


In [28]:
example = movies_dedup.loc[
    movies_dedup["title"].isin(["Inception", "Interstellar"]),
    ["id", "title"],
].copy()

example["semantic_text"] = semantic_text.loc[example.index]
example

,id,title,semantic_text
0,27205,Inception,"Overview: Cobb, a skilled thief who commits co..."
1,157336,Interstellar,Overview: The adventures of a group of explore...


Baris dengan `semantic_text` kosong seharusnya berasal dari film yang ketiga kolom
sumbernya juga kosong. Pemeriksaan ini memastikan tidak ada teks yang hilang karena
kesalahan penggabungan.

Film seperti ini tetap disimpan di master dataset, tetapi tidak akan memperoleh
embedding.


In [29]:
empty_examples = movies_dedup.loc[
    semantic_text.eq(""),
    ["id", "title", "overview", "tagline", "keywords"],
].head(5)

empty_examples

,id,title,overview,tagline,keywords
10072,919952,Il grande giorno,NaN,NaN,NaN
10867,17413,Incognito,NaN,NaN,NaN
11050,48805,L'allenatore nel pallone 2,NaN,NaN,NaN
11177,77000,Bar Sport,NaN,NaN,NaN
11398,57114,"Amore, bugie e calcetto",NaN,NaN,NaN


## 10. Load Model dan Audit Panjang Token

Model yang dipakai adalah SBERT dengan backbone DistilBERT. Model memetakan teks ke
ruang vektor 768 dimensi dan menghasilkan embedding yang sudah dinormalisasi L2.

`max_seq_length` dibaca langsung dari konfigurasi model, bukan diasumsikan. Nilai
ini menjadi batas jumlah token per teks saat encoding, dan menentukan apakah ada
teks yang akan terpotong.

Model ini dipakai sebagai **feature extractor**. Bobotnya tidak diubah, tidak ada
loss, optimizer, atau backpropagation pada seluruh pipeline.


In [30]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/multi-qa-distilbert-cos-v1"

model = SentenceTransformer(MODEL_NAME, device="cuda")
tokenizer = model[0].tokenizer
max_seq_length = model.max_seq_length

print("model:", MODEL_NAME)
print("max sequence length:", max_seq_length)
print("tokenizer:", tokenizer.name_or_path)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

model: sentence-transformers/multi-qa-distilbert-cos-v1
max sequence length: 512
tokenizer: sentence-transformers/multi-qa-distilbert-cos-v1


Hanya teks yang memiliki karakter non-whitespace yang diteruskan ke tokenisasi.
Film dengan `semantic_text` kosong tetap berada di master dataset, tetapi tidak
punya representasi semantik.

Hasilnya 396.254 teks, sama dengan angka pada bagian ketersediaan teks.


In [31]:
eligible_mask = semantic_text.notna() & semantic_text.str.strip().ne("")
texts = semantic_text.loc[eligible_mask]

print("texts to inspect:", len(texts))

texts to inspect: 396254


Panjang diukur memakai tokenizer model, bukan jumlah kata. Satu kata dapat dipecah
menjadi beberapa subword, dan token khusus seperti `[CLS]` serta `[SEP]` ikut
dihitung.

`truncation=False` sengaja dipakai agar yang terukur adalah panjang penuh.
Peringatan tokenizer bahwa sebuah sequence melebihi 512 token bukan error, melainkan
informasi bahwa teks tersebut akan terpotong saat encoding.


In [32]:
TOKEN_BATCH_SIZE = 2048
token_lengths = []

for start in range(0, len(texts), TOKEN_BATCH_SIZE):
    batch = texts.iloc[start : start + TOKEN_BATCH_SIZE].tolist()

    encoded = tokenizer(
        batch,
        add_special_tokens=True,
        truncation=False,
        padding=False,
        return_length=True,
    )

    token_lengths.extend(encoded["length"])

token_lengths = pd.Series(
    token_lengths,
    index=texts.index,
    name="token_length",
)

print("measured texts:", len(token_lengths))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (558 > 512). Running this sequence through the model will result in indexing errors


measured texts: 396254


Persentil ke-90, 95, dan 99 lebih informatif daripada nilai maksimum, karena
maksimum mudah dipengaruhi satu record ekstrem.

Median menunjukkan panjang teks film tipikal. Selisih antara median dan persentil
ke-99 menunjukkan seberapa panjang ekor distribusinya.


In [33]:
token_lengths.describe(percentiles=[0.50, 0.90, 0.95, 0.99])

count    396254.000000
mean         72.914252
std          46.976944
min           5.000000
50%          61.000000
90%         140.000000
95%         168.000000
99%         215.000000
max         782.000000
Name: token_length, dtype: float64

Batas keras model adalah `max_seq_length`. Teks yang lebih panjang akan dipotong
saat encoding, sehingga bagian setelah batas tidak ikut membentuk embedding.

Karena urutan teks adalah overview, tagline, lalu keywords, bagian akhir berisiko
terpotong lebih dulu.


In [34]:
n_over_limit = int(token_lengths.gt(max_seq_length).sum())
pct_over_limit = (
    n_over_limit / len(token_lengths) * 100
    if len(token_lengths)
    else 0.0
)

print("max sequence length:", max_seq_length)
print("texts exceeding limit:", n_over_limit)
print("percentage exceeding limit:", f"{pct_over_limit:.4f}%")

max sequence length: 512
texts exceeding limit: 16
percentage exceeding limit: 0.0040%


Tabel di bawah menunjukkan film dengan teks terpanjang beserta jumlah tokennya.
Cuplikan teks membantu memeriksa bagian mana yang berpotensi terpotong.

Kalau jumlah teks yang melewati batas sangat kecil, truncation bisa dilaporkan
sebagai keterbatasan minor tanpa mengubah strategi encoding.


In [35]:
over_limit_lengths = token_lengths[token_lengths > max_seq_length]
long_indices = over_limit_lengths.sort_values(ascending=False).head(10).index

long_examples = movies_dedup.loc[
    long_indices,
    ["id", "title"],
].copy()

long_examples["token_length"] = token_lengths.loc[long_indices]
long_examples["semantic_text_preview"] = (
    semantic_text.loc[long_indices].str.slice(0, 500)
)

long_examples

,id,title,token_length,semantic_text_preview
211503,77994,Yegor's Grief,782,Overview: У Егора проблемы. Он занял денег для...
213506,64060,Sokrovishche,760,Overview: Это – новогодняя комедия-сказка. Нев...
211238,77831,This better not to know,745,"Overview: Говорить о том, что ревность - это ч..."
217837,70139,Kogda My Byli Schastlivy,735,Overview: В жизни бизнесмена Глеба все было ид...
181498,64380,Nezhnost K Revushemu Zveryu,691,Overview: По мотивам одноименного романа А. Ба...
199542,73054,Super-Mother-in-Law For Loser,660,Overview: Леонид - злоумышленник средней руки....
321167,64184,Volga River Flows,648,Overview: В заброшенный поселок на берегу Волг...
218182,92735,Padeniye Kondora,647,Overview: Действие происходит в вымышленном го...
218976,70945,Dzisay,621,"Overview: ""Дзисай"" в переводе с древнеяпонског..."
94234,71047,First Encounter - Last Encounter,607,Overview: История о событиях в Петербурге в ка...


Model card menyebut model dilatih pada teks hingga sekitar 250 word piece. Angka ini
bukan batas keras, tetapi teks yang jauh lebih panjang berada di luar rentang yang
paling banyak dilihat model saat training.

Perbandingan ini dicatat sebagai konteks, bukan alasan mengubah batas model.


In [36]:
training_reference_length = 250

n_over_training_reference = int(
    token_lengths.gt(training_reference_length).sum()
)
pct_over_training_reference = (
    n_over_training_reference / len(token_lengths) * 100
)

print("texts over 250 tokens:", n_over_training_reference)
print(
    "percentage over 250 tokens:",
    f"{pct_over_training_reference:.2f}%",
)

texts over 250 tokens: 520
percentage over 250 tokens: 0.13%


Kesimpulan dari audit panjang token:

1. Tidak ada teks yang ekstrem panjangnya dibanding batas model.
2. Jumlah teks yang melewati `max_seq_length` sangat kecil dibanding total.
3. Teks yang melewati 250 token berada di luar rentang training, tetapi tetap
   diproses utuh selama masih di bawah 512.

Keputusan yang dipakai: batas `max_seq_length` dibiarkan apa adanya, mengikuti
konfigurasi model, dan teks yang terpotong dilaporkan sebagai keterbatasan.

Encoding SBERT untuk seluruh teks tidak dilakukan di notebook ini, melainkan di
`03_semantic_feature.ipynb` dan `scripts/02_generate_sbert_embeddings.py`.


## 11. Menyimpan Hasil Preprocessing

Dua artifact disimpan di sini. `semantic_text.parquet` memuat 429.755 baris, satu
untuk setiap film di master dataset, termasuk yang teksnya kosong. Sedangkan
`token_lengths.parquet` hanya memuat 396.254 baris, yaitu teks yang benar-benar
diukur.

Perbedaan jumlah baris ini disengaja. File pertama dipakai untuk melihat seluruh
populasi, file kedua dipakai untuk keperluan encoding.

Keduanya disimpan ke folder baseline. Dataset final punya artifact sendiri di
`artifacts/semantic/`, yang dibangun di notebook 03.


In [39]:
from movie_recommender.utils.paths import RESULTS_DIR, SEMANTIC_BASELINE_ARTIFACTS_DIR

SEMANTIC_BASELINE_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

semantic_text_frame = pd.DataFrame(
    {
        "id": movies_dedup["id"],
        "semantic_text": semantic_text,
    }
)
semantic_text_frame.to_parquet(
    SEMANTIC_BASELINE_ARTIFACTS_DIR / "semantic_text.parquet",
    index=False,
    compression="zstd",
)

token_length_frame = pd.DataFrame(
    {
        "id": movies_dedup.loc[texts.index, "id"],
        "token_length": token_lengths,
    }
).reset_index(drop=True)
token_length_frame.to_parquet(
    SEMANTIC_BASELINE_ARTIFACTS_DIR / "token_lengths.parquet",
    index=False,
    compression="zstd",
)

print("semantic_text rows:", len(semantic_text_frame))
print("token_length rows:", len(token_length_frame))

semantic_text rows: 429755
token_length rows: 396254


File dibaca kembali untuk memastikan bentuk dan tipenya sesuai. Langkah ini murah,
tetapi mencegah masalah yang sulit dilacak, misalnya kolom yang berubah tipe setelah
disimpan.


In [40]:
loaded_text = pd.read_parquet(
    SEMANTIC_BASELINE_ARTIFACTS_DIR / "semantic_text.parquet"
)
loaded_lengths = pd.read_parquet(
    SEMANTIC_BASELINE_ARTIFACTS_DIR / "token_lengths.parquet"
)

print(loaded_text.shape, loaded_lengths.shape)
print(loaded_text.head())
print(loaded_lengths.head())

(429755, 2) (396254, 2)
       id                                      semantic_text
0   27205  Overview: Cobb, a skilled thief who commits co...
1  157336  Overview: The adventures of a group of explore...
2     155  Overview: Batman raises the stakes in his war ...
3   19995  Overview: In the 22nd century, a paraplegic Ma...
4   24428  Overview: When an unexpected enemy emerges and...
       id  token_length
0   27205           123
1  157336           153
2     155           147
3   19995            97
4   24428           117


## 12. Dataset Eksperimen

Dataset eksperimen adalah irisan antara master dataset dan film yang punya teks.

Urutannya sengaja mengikuti `movie_ids.npy`, bukan urutan master dataset. Alasannya,
baris ke-i pada matriks embedding harus merujuk film yang sama dengan baris ke-i
pada dataset ini. Tanpa penyeragaman urutan, penggabungan berdasarkan posisi baris
akan salah.

Cell ini memeriksa tiga hal sebelum menyimpan: ID unik, jumlah baris sama dengan
jumlah ID embedding, dan urutan ID identik dengan `movie_ids.npy`.


In [26]:
import numpy as np
import pandas as pd

from movie_recommender.utils.paths import (
    DATA_PROCESSED_DIR,
    MOVIES_PARQUET,
    SEMANTIC_BASELINE_ARTIFACTS_DIR,
)

# 1. Sumber kebenaran urutan subset
movie_ids = np.load(SEMANTIC_BASELINE_ARTIFACTS_DIR / "movie_ids.npy")

# 2. Master dan teks semantik
movies = pd.read_parquet(MOVIES_PARQUET).set_index("id")
text_frame = pd.read_parquet(
    SEMANTIC_BASELINE_ARTIFACTS_DIR / "semantic_text.parquet"
).set_index("id")

# 3. Validasi
if not movies.index.is_unique:
    raise ValueError("Master dataset contains duplicate movie IDs")

if not text_frame.index.is_unique:
    raise ValueError("Semantic text artifact contains duplicate movie IDs")

missing_movies = set(movie_ids) - set(movies.index)
missing_texts = set(movie_ids) - set(text_frame.index)

if missing_movies:
    raise ValueError(f"Missing movie IDs in master: {len(missing_movies)}")
if missing_texts:
    raise ValueError(f"Missing movie IDs in semantic text: {len(missing_texts)}")

# 4. Ambil subset dengan urutan yang sama seperti embedding
experiment = movies.loc[movie_ids].copy()
experiment["semantic_text"] = text_frame.loc[
    movie_ids, "semantic_text"
].to_numpy()

experiment = experiment.reset_index()

# 5. Verifikasi alignment
if not np.array_equal(
    experiment["id"].to_numpy(),
    movie_ids,
):
    raise ValueError("Experiment dataset is not aligned with movie_ids")

if len(experiment) != len(movie_ids):
    raise ValueError("Row count mismatch")

print("shape:", experiment.shape)
print("unique ids:", experiment["id"].is_unique)
print("semantic text empty:", int(experiment["semantic_text"].eq("").sum()))

# 6. Simpan
output_path = DATA_PROCESSED_DIR / "movies_experiment.parquet"
output_path.parent.mkdir(parents=True, exist_ok=True)

experiment.to_parquet(
    output_path,
    index=False,
    compression="zstd",
)

print("saved:", output_path)

shape: (396254, 30)
unique ids: True
semantic text empty: 0
saved: D:\College Life\Skripsiw\hybrid-movie-recommender\data\processed\movies_experiment.parquet


## 13. Finalisasi Dataset Pemodelan

Bagian ini membuang film yang tidak punya metadata sama sekali, lalu menyimpan
dataset yang siap dipakai untuk pemodelan hybrid.

### Alasan

Penelitian ini membandingkan penggabungan dua sumber sinyal, yaitu teks semantik
dan metadata. Film yang keempat kolom metadatanya kosong tidak dapat menjadi bagian
dari perbandingan tersebut.

Kalau film seperti itu dipertahankan, vektor metadatanya akan bernilai nol semua.
Normanya nol, sehingga cosine similarity tidak terdefinisi. Dalam praktik, film
tersebut akan selalu mendapat skor metadata nol dan tidak pernah muncul pada kondisi
`alpha` rendah. Itu menciptakan bias sistematis pada evaluasi.

### Aturan

Film dipertahankan jika **minimal satu** dari `genres`, `directors`, `writers`, atau
`cast` berisi teks. Film dibuang hanya jika keempatnya kosong.

### Konsekuensi

Dataset turun dari 388.241 menjadi 387.188 baris, atau 0,271 persen. Film yang
dibuang bukan film populer: median `vote_count` mereka nol, dan median tahun
rilisnya 1930. Sebagian besar berupa newsreel, film bisu, dan entri TMDB dengan data
sangat minim.

Daftar lengkap film yang dibuang disimpan ke
`data/processed/excluded_no_metadata.csv`, supaya keputusan ini dapat diaudit dan
dilampirkan pada laporan.

### Urutan pengerjaan

Bagian ini membutuhkan `movies_final.parquet`, yang dihasilkan oleh notebook 02.
Jadi bagian ini dijalankan **setelah** notebook 02 selesai, bukan sebelum.


In [2]:
import numpy as np
import pandas as pd

from movie_recommender.utils.paths import DATA_PROCESSED_DIR

SOURCE = DATA_PROCESSED_DIR / "movies_final.parquet"
TARGET = DATA_PROCESSED_DIR / "movies_modeling.parquet"
EXCLUDED = DATA_PROCESSED_DIR / "excluded_no_metadata.csv"

METADATA_COLUMNS = ["genres", "directors", "writers", "cast"]

movies = pd.read_parquet(SOURCE)
print("sumber:", movies.shape)

# Aturan wajib
assert movies["id"].is_unique, "ID duplikat"
assert movies["id"].notna().all(), "ID kosong"

overview = movies["overview"].fillna("").astype(str).str.strip()
semantic = movies["semantic_text"].fillna("").astype(str).str.strip()

assert overview.ne("").all(), "ada overview kosong"
assert semantic.ne("").all(), "ada semantic_text kosong"

# Kelengkapan metadata: minimal satu kolom terisi
filled = pd.DataFrame(
    {
        column: movies[column].fillna("").astype(str).str.strip().ne("")
        for column in METADATA_COLUMNS
    }
)
has_metadata = filled.any(axis=1)

kept = movies.loc[has_metadata].reset_index(drop=True)
excluded = movies.loc[~has_metadata].reset_index(drop=True)

print("dengan metadata:", len(kept))
print("tanpa metadata :", len(excluded))
print("total          :", len(kept) + len(excluded), "dari", len(movies))

sumber: (388241, 31)
dengan metadata: 387188
tanpa metadata : 1053
total          : 388241 dari 388241


In [3]:
exclude_columns = [
    column
    for column in ["id", "title", "release_date", "original_language", "vote_count"]
    if column in excluded.columns
]

excluded_frame = excluded[exclude_columns].copy()
excluded_frame["reason"] = "tidak ada metadata terisi"
excluded_frame.to_csv(EXCLUDED, index=False, encoding="utf-8-sig")

kept.to_parquet(TARGET, index=False, compression="zstd")

print("dibuang :", EXCLUDED, "|", len(excluded_frame), "baris")
print("disimpan:", TARGET, "|", len(kept), "baris")
print("ukuran MB:", round(TARGET.stat().st_size / 1024**2, 1))

dibuang : D:\College Life\Skripsiw\hybrid-movie-recommender\data\processed\excluded_no_metadata.csv | 1053 baris
disimpan: D:\College Life\Skripsiw\hybrid-movie-recommender\data\processed\movies_modeling.parquet | 387188 baris
ukuran MB: 170.7


In [5]:
check = pd.read_parquet(TARGET)

# 1. Round trip
pd.testing.assert_frame_equal(check, kept)

# 2. Bentuk dan skema
print("shape:", check.shape)
assert check.shape[1] == movies.shape[1], "jumlah kolom berubah"
assert check["id"].is_unique

# 3. Tidak ada yang hilang atau bertambah
assert len(check) + len(excluded) == len(movies)
assert set(check["id"]).issubset(set(movies["id"]))

# 4. Aturan wajib tetap berlaku
assert check["overview"].fillna("").astype(str).str.strip().ne("").all()
assert check["semantic_text"].fillna("").astype(str).str.strip().ne("").all()
assert check["language_status"].eq("non_english").sum() == 0

# 5. Semua punya metadata sekarang
still_empty = pd.DataFrame(
    {
        column: check[column].fillna("").astype(str).str.strip().ne("")
        for column in METADATA_COLUMNS
    }
).any(axis=1)
print("film tanpa metadata setelah filter:", int((~still_empty).sum()))
assert still_empty.all()

# 6. semantic_text tidak boleh berubah
source_text = movies.set_index("id")["semantic_text"]
target_text = check.set_index("id")["semantic_text"]
print(
    "semantic_text identik:",
    bool(
        np.array_equal(
            source_text.loc[target_text.index].to_numpy(),
            target_text.to_numpy(),
        )
    ),
)

print()
print("semua pemeriksaan lulus")

shape: (387188, 31)
film tanpa metadata setelah filter: 0
semantic_text identik: True

semua pemeriksaan lulus


Pemeriksaan di atas mencakup enam hal:

| Pemeriksaan | Tujuan |
|---|---|
| Round trip `assert_frame_equal` | Memastikan file tersimpan sama dengan objek di memori |
| Jumlah kolom | Memastikan tidak ada kolom yang hilang saat penyimpanan |
| Jumlah baris | Memastikan yang disimpan ditambah yang dibuang sama dengan total awal |
| Subset ID | Memastikan tidak ada ID baru yang muncul |
| Aturan wajib | Memastikan tidak ada overview, `semantic_text`, atau `non_english` yang lolos |
| Teks tidak berubah | Memastikan filter hanya membuang baris, tidak menyunting teks |

Pemeriksaan terakhir yang paling penting. Kalau `semantic_text` berubah, embedding
yang sudah dihitung tidak lagi berlaku, dan encoding harus diulang.


## 14. Ringkasan Notebook

Notebook ini menghasilkan tiga hal.

Pertama, master dataset berisi 429.755 film unik, terbentuk dari 443.610 baris
mentah setelah deduplikasi berbasis kelengkapan metadata.

Kedua, `semantic_text` untuk seluruh film di master dataset, beserta audit panjang
token yang menunjukkan bahwa teks pada dataset ini pendek dan hanya sedikit yang
mendekati batas model.

Ketiga, dua dataset turunan: dataset eksperimen berisi 396.254 film yang punya teks,
dan dataset pemodelan berisi 387.188 film yang punya teks sekaligus metadata.

Angka yang bisa dikutip:

| Item | Nilai |
|---|---|
| Baris mentah | 443.610 |
| ID unik | 429.755 |
| Grup ID duplikat | 3.463 |
| Film dengan teks | 396.254 |
| Film tanpa teks | 33.501 |
| Film tanpa teks tetapi punya metadata | 33.405 |
| Film tanpa teks dan tanpa metadata | 96 |
| Film yang dibuang karena metadata kosong | 1.053 |
| Dataset pemodelan final | 387.188 |

Tahap berikutnya adalah filter bahasa di notebook 02, lalu pembentukan fitur
semantik dan encoding di notebook 03.
